<a href="https://colab.research.google.com/github/gauravd12345/wikiGPT/blob/main/wikiGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [73]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

device: cpu


In [10]:
!pip install convokit
from convokit import Corpus, download

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.9/240.9 kB 6.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.4 MB/s eta 0:00:00
  Created wheel for convokit: filename=convokit-4.1.1-py3-none-any.whl size=292184 sha256=137d707829285029ad2b99dcd9251ca408a7a08080ffae9387a154223f039dcb
  Stored in directory: /root/.cache/pip/wheels/f8/87/6a/02d40d2aac8d827060bf

In [11]:
corpus = Corpus(filename=download("movie-corpus"))

No configuration file found at /root/.convokit/config.yml; writing with contents: 
# Default Backend Parameters
db_host: localhost:27017
data_directory: ~/.convokit/saved-corpora
model_directory: ~/.convokit/saved-models
default_backend: mem


In [12]:
corpus.print_summary_stats()

Number of Speakers: 9035
Number of Utterances: 304713
Number of Conversations: 83097


In [25]:
utt = corpus.random_utterance()

print("ID:", utt.id, "\n")
print("Reply_to:", utt.reply_to, "\n")
print("Timestamp:", utt.timestamp, "\n")
print("Text:", utt.text, "\n")
print("Conversation ID:", utt.conversation_id, "\n")
print("Speaker ID:", utt.speaker.id)


ID: L83461 

Reply_to: L83460 

Timestamp: None 

Text: They'll know. They'll know as soon as they born. Cause it's inside us,Stamp. It'll be inside them. We'll pass it down. Schoolteacher didn't just change the outside, he changed the mind..and the blood..and what it carries...and what it's worth.. 

Conversation ID: L83455 

Speaker ID: u4043


In [47]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.add_special_tokens({"additional_special_tokens": ["<sep>"]})

1

In [53]:
buffer = []

for convo in corpus.iter_conversations():
    utterances = get_ordered_utterances(convo)
    texts = [utt.text.strip() for utt in utterances if utt.text and utt.text.strip()]
    if len(texts) < 2:
        continue
    buffer.extend(tokenizer.encode(" <sep> ".join(texts) + " <eos>"))

block_size = 64
chunks = [buffer[i : i + block_size + 1] for i in range(0, len(buffer) - block_size, block_size)]


X = torch.tensor([c[:-1] for c in chunks], dtype=torch.long)
y = torch.tensor([c[1:]  for c in chunks], dtype=torch.long)

print(X.shape, y.shape)

torch.Size([81905, 64]) torch.Size([81905, 64])


In [71]:
text = tokenizer.decode(X[1])
text2 = tokenizer.decode(y[1])
print(text)
print(text2)

? <sep> No <sep> Okay -- you're gonna need to learn how to lie. <eos>I figured you'd get to the good stuff eventually. <sep> What good stuff? <sep> The "real you". <sep> Like my fear of wearing pastels? <eos>do you listen to
 <sep> No <sep> Okay -- you're gonna need to learn how to lie. <eos>I figured you'd get to the good stuff eventually. <sep> What good stuff? <sep> The "real you". <sep> Like my fear of wearing pastels? <eos>do you listen to this


In [57]:
""" Hyper parameter """

d_model = 512
h = 8
d_k = 64
d_v = 64
N = 6

lr = 0.001
epochs = 10
batch_size = 64
vocab_size = tokenizer.vocab_size

50257


In [56]:
class DecoderTransformerDataset(Dataset):
  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

  def __len__(self):
    return len(self.X)

In [66]:
class DecoderTransformer(nn.Module):
  def __init__(self):
    super().__init__()

    self.embed = nn.Embedding(vocab_size, d_model)
    self.pos = nn.Embedding(block_size, d_model) # positional embedding

    self.W_q = nn.ModuleList([nn.Linear(d_model, d_k) for _ in range(h)]) # q, k, v projections
    self.W_k = nn.ModuleList([nn.Linear(d_model, d_k) for _ in range(h)])
    self.W_v = nn.ModuleList([nn.Linear(d_model, d_v) for _ in range(h)])

    self.W_o = nn.Linear(h * d_v, d_model) # final projection

    self.ffn1 = nn.Linear(d_model, d_model) # ffn layer
    self.ffn2 = nn.Linear(d_model, d_model)

    self.fc = nn.Linear(d_model, vocab_size) # final layer

    self.ln1 = nn.LayerNorm(d_model)
    self.ln2 = nn.LayerNorm(d_model)

  def multi_head_attention(self, x):
    W_tot = []
    mask = torch.triu(torch.full((d_model, d_model), float('-inf')), diagonal=1)
    for Q, K, V in zip(self.W_q, self.W_k, self.W_v):
      Q_i = Q(x)
      K_i = K(x)
      V_i = V(x)

      alignment = torch.matmul(Q_i * K_i.mt)                       # query-key alignment
      alignment = alignment + mask                                 # adding mask

      wei = torch.softmax(alignment / (d_k ** 0.5), dim=1)         # alignment weights
      wei_value = torch.matmul(wei, V_i)                           # weighted values

      W_tot.append(wei_value)

    out = self.W_o(torch.cat(wei_value, dim=1))
    return out

  def forward(self, x): # (batch_size, block_size)
    p = torch.arange(block_size)
    x = self.embed(x) + self.pos(p)     # word & positional embedding

    for _ in range(N):
      out = self.multi_head_attention(x)  # multi head attention
      out = self.ln1(out + x)             # layernorm + residual connection

      fn = self.ffn1(out)                 # ffn
      fn = torch.relu(fn)
      fn = self.ffn2(fn)

      out = self.ln2(fn + out)
      x = out

    return out


In [70]:
dataset = DecoderTransformerDataset(X, y)
dataloader = DataLoader(
    dataset=dataset,
    batch_size=batch_size
)

model = DecoderTransformer()

In [82]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

model.train()
for epoch in range(epochs):
  total_loss = 0.0
  for xb, yb in dataloader:
    optimizer.zero_grad()

    xb = xb.to(device)
    yb = yb.to(device)
    print(xb, vocab_size)
    out = model(xb)
    loss = criterion(out, yb)
    total_loss += loss

    loss.backward()
    optimizer.step()

  print(f"Epoch: {epoch} | Loss: {total_loss / batch_size}")


tensor([[ 2990,   466,   284,  ...,   703,   284, 11238],
        [   30,   220, 50257,  ...,   345,  6004,   284],
        [  428, 18824,    30,  ...,   467,   597, 14871],
        ...,
        [  996,    13,   220,  ...,  1101,   261,    13],
        [  220,   314,  1101,  ...,  4252,    82,   764],
        [  220,  1320,   338,  ...,    34, 41639,  1377]]) 50257


IndexError: index out of range in self